**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion Models

The generative method behind modern image synthesis, told as a *signal processing* story: corrupt data with Gaussian noise step by step, train a network to **denoise**, then run the corruption in reverse. We train a complete diffusion model on 2-D data in minutes and watch noise crystallize into structure.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).
- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (Gaussians compose).
- [Representation Learning](./Representation_Learning.ipynb) S2 for the generative-model context.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# target distribution: two moons — structured, multimodal, low-dimensional
def moons(n):
    t = rng.uniform(0, np.pi, n)
    top = np.stack([np.cos(t), np.sin(t)], 1)
    bot = np.stack([1 - np.cos(t), 0.4 - np.sin(t)], 1)
    X = np.concatenate([top[:n//2], bot[n//2:]]) + 0.06*rng.standard_normal((n, 2))
    return ((X - X.mean(0)) / X.std(0)).astype(np.float32)

X = torch.from_numpy(moons(6000))
plt.figure(figsize=(3.6, 3.2)); plt.scatter(*X.T, s=2, alpha=0.3)
plt.title("the distribution we want to SAMPLE from"); plt.axis("equal")
plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 2 — *The Forward Process & the Denoising Objective* (~35 min)
**Goal:** destroy data with scheduled noise; train a network to predict the noise.
**Feeds into:** Session 2 (sampling = reverse diffusion).

---

## 2. Destruction Is Easy — Learn to Undo It

💡 **Intuition.** Generating from scratch is hard; *removing a little noise* is easy — it's [Wiener denoising's](../Intro_DSP/Statistical_Signal_Processing.ipynb) cousin, a regression problem. Diffusion's insight: chain the easy problem. Define a forward process that gradually noises data into pure Gaussian ($x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ — Gaussians compose, so any step is one formula), and train one network $\varepsilon_\theta(x_t, t)$ to **predict the noise** that was added. Denoising at *every* noise level = knowing the path from chaos back to data.

In [ ]:
# visualize the forward death of the data

# YOUR CODE HERE


In [ ]:
# the denoiser: predicts ε from (x_t, t) — t is embedded sinusoidally, like a transformer position

# YOUR CODE HERE


---
### 🕐 Session 2 of 2 — *Sampling: Running Time Backwards* (~40 min)
**Goal:** start from pure noise and iteratively denoise into fresh samples.
**Builds on:** Session 1.

---

## 3. The Reverse Process

💡 **Intuition.** To sample: start at $x_T \sim \mathcal{N}(0, I)$ and repeatedly apply the learned denoiser, stepping $t = T{-}1, \dots, 0$, re-injecting a *little* fresh noise each step (the stochasticity keeps samples diverse — drop it and you get DDIM's deterministic cousin). Each step is a small, easy denoise; a few hundred of them compound into creation. Image generators are this exact loop with a U-Net denoiser and billions of pixels.

In [ ]:

# YOUR CODE HERE


In [ ]:
# quantitative check: do generated samples match the data's statistics?
# and the harder test: fraction of generated points close to the true manifold

# YOUR CODE HERE


**The DSP lens, explicitly:** the forward process is progressive low-pass-plus-noise (coarse structure survives longest); the reverse process therefore builds coarse structure first and details last — generation as *spectral refinement*. That's also why diffusion models are natural denoisers, inpainters, and super-resolvers: those are just partial trips along the same chain.

## 4. Conclusion

One regression loss (predict the noise), one schedule, and a walk backwards through it: that's the entire method behind modern generative imagery — and you just trained one.

---
## Where next

- [Representation Learning](./Representation_Learning.ipynb) — VAEs: the previous generation of generation.
- [Statistical Signal Processing](../Intro_DSP/Statistical_Signal_Processing.ipynb) — the denoising theory underneath.
- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — what it takes to run this at image scale.